# 04 – Data Preprocessing and Feature Engineering

## 1. Introduction

This notebook documents the formal preprocessing and feature engineering workflow for the UCI Online Retail dataset, preparing customer-level observations for unsupervised customer segmentation.

The work builds on the dataset understanding, data-quality assessment and exploratory analysis completed in Notebooks 01–03.

The primary objective is to construct a merchandise-only customer dataset and derive two principal feature representations:

- **RFM features:** Recency, Frequency and Monetary.
- **Extended behavioural features:** RFM plus Average Order Value, Total Quantity Purchased and Product Diversity.

Purchase Regularity is calculated and retained for descriptive assessment, but is excluded from clustering because it is undefined for customers with only one invoice. No imputation is applied.

## 2. Preprocessing principles

- Preserve the original raw dataset unchanged.
- Apply all cleaning and transformations to separate working DataFrames.
- Exclude records without an identifiable CustomerID from customer-level clustering.
- Separate cancellation records from purchase records.
- Investigate zero/negative transaction values before defining eligible purchases.
- Retain genuine merchandise StockCodes, including alphanumeric identifiers.
- Separate postage and manual codes from primary merchandise features.
- Investigate influential observations rather than automatically deleting or capping statistical outliers.
- Validate each stage and save processed datasets for reproducible modelling.

## 3. Workflow

The notebook proceeds from transaction-level cleaning to customer-level aggregation, feature transformation, correlation and outlier assessment, sensitivity preparation, scaling, validation and export.


### Section 4.1 – Load data and establish preprocessing environment

In [1]:

# ============================================================
# 4.1 LOAD ORIGINAL DATA AND SET UP PREPROCESSING
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_online_retail

# Load untouched original dataset
raw_df = load_online_retail()

# Create independent working copy
working_df = raw_df.copy(deep=True)

# Standardise invoice and stock-code text for filtering
working_df["InvoiceNo"] = (
    working_df["InvoiceNo"].astype("string").str.strip()
)

working_df["StockCode"] = (
    working_df["StockCode"].astype("string").str.strip()
)

print("Original dataset:", raw_df.shape)
print("Working dataset:", working_df.shape)
print("Original data preserved:", raw_df.equals(
    load_online_retail()
))


Original dataset: (541909, 8)
Working dataset: (541909, 8)
Original data preserved: True


**Interpretation**

The original transaction file is loaded without modification, and a separate working copy is created for preprocessing. This preserves an auditable raw source while allowing standardisation of invoice and stock-code text in the analytical copy.

### Section 4.2 – Remove exact duplicate records

In [2]:

# ============================================================
# 4.2 REMOVE EXACT DUPLICATE RECORDS
# ============================================================

# Preserve duplicate records separately for audit
duplicate_records = working_df[
    working_df.duplicated(keep="first")
].copy()

# Remove excess exact duplicates
deduplicated_df = working_df.drop_duplicates(
    keep="first"
).copy()

print("BEFORE DUPLICATE REMOVAL:", len(working_df))
print("Duplicate rows removed:", len(duplicate_records))
print("AFTER DUPLICATE REMOVAL:", len(deduplicated_df))

print(
    "Remaining exact duplicates:",
    deduplicated_df.duplicated().sum()
)


BEFORE DUPLICATE REMOVAL: 541909
Duplicate rows removed: 5268
AFTER DUPLICATE REMOVAL: 536641
Remaining exact duplicates: 0


**Interpretation**

Exact duplicate rows are removed from the working copy, while duplicate records are retained separately for audit. This step removes repeated records but cannot establish whether every repeated transaction line is erroneous, because the source does not provide a unique transaction-line identifier.

### Section 4.3 – Separate cancellations and purchase transactions

In [3]:

# ============================================================
# 4.3 SEPARATE CANCELLATIONS FROM PURCHASE TRANSACTIONS
# ============================================================

invoice_text = deduplicated_df["InvoiceNo"].astype("string").str.strip()

cancellation_mask = invoice_text.str.startswith(
    "C", na=False
)

# Preserve cancellation records
cancellation_records = deduplicated_df.loc[
    cancellation_mask
].copy()

# Keep non-cancellation records for further preprocessing
purchase_candidates = deduplicated_df.loc[
    ~cancellation_mask
].copy()

print("CANCELLATION SEPARATION")
print("Deduplicated records:", len(deduplicated_df))
print("Cancellation records:", len(cancellation_records))
print("Non-cancellation records:", len(purchase_candidates))

print(
    "\nDistinct cancellation invoices:",
    cancellation_records["InvoiceNo"].nunique()
)

print(
    "Distinct non-cancellation invoices:",
    purchase_candidates["InvoiceNo"].nunique()
)

print(
    "\nValidation:",
    len(cancellation_records) + len(purchase_candidates)
    == len(deduplicated_df)
)


CANCELLATION SEPARATION
Deduplicated records: 536641
Cancellation records: 9251
Non-cancellation records: 527390

Distinct cancellation invoices: 3836
Distinct non-cancellation invoices: 22064

Validation: True


**Interpretation**

Cancellation invoices are separated from non-cancellation purchase candidates using the `C` invoice prefix. The validation confirms that the two subsets account for all deduplicated records. Cancellation records are retained for investigation rather than treated as completed purchases.

### Section 4.4 – Separate records with missing CustomerID

In [4]:
# Separate non-cancellation records based on CustomerID availability

missing_customer_id = purchase_candidates[
    purchase_candidates["CustomerID"].isna()
].copy()

identified_customer_records = purchase_candidates[
    purchase_candidates["CustomerID"].notna()
].copy()

print("CUSTOMER ID SEPARATION")
print("Non-cancellation records:", len(purchase_candidates))
print("Records with missing CustomerID:", len(missing_customer_id))
print("Records with identified CustomerID:", len(identified_customer_records))

print(
    "\nValidation:",
    len(missing_customer_id) + len(identified_customer_records)
    == len(purchase_candidates)
)

CUSTOMER ID SEPARATION
Non-cancellation records: 527390
Records with missing CustomerID: 134658
Records with identified CustomerID: 392732

Validation: True


**Interpretation**

Records without CustomerID cannot be assigned to an individual customer and are excluded from customer-level segmentation. Both groups are retained separately so the exclusion is transparent and quantifiable.

### Section 4.5 – Separate invalid transaction values

In [5]:
# Separate identified-customer records by transaction validity

negative_quantity = identified_customer_records[
    identified_customer_records["Quantity"] < 0
].copy()

zero_quantity = identified_customer_records[
    identified_customer_records["Quantity"] == 0
].copy()

negative_price = identified_customer_records[
    identified_customer_records["UnitPrice"] < 0
].copy()

zero_price = identified_customer_records[
    identified_customer_records["UnitPrice"] == 0
].copy()

valid_positive_transactions = identified_customer_records[
    (identified_customer_records["Quantity"] > 0)
    & (identified_customer_records["UnitPrice"] > 0)
].copy()

print("TRANSACTION VALUE CHECK")
print("Identified-customer records:", len(identified_customer_records))
print("Negative quantity:", len(negative_quantity))
print("Zero quantity:", len(zero_quantity))
print("Negative unit price:", len(negative_price))
print("Zero unit price:", len(zero_price))
print("Valid positive transactions:", len(valid_positive_transactions))

print("\nValidation:")
print(
    "Valid positive transactions:",
    len(valid_positive_transactions)
)

TRANSACTION VALUE CHECK
Identified-customer records: 392732
Negative quantity: 0
Zero quantity: 0
Negative unit price: 0
Zero unit price: 40
Valid positive transactions: 392692

Validation:
Valid positive transactions: 392692


**Interpretation**

The primary customer analysis requires positive quantities and positive unit prices. Negative quantities, zero quantities, negative prices and zero prices are isolated rather than silently combined with valid purchases.

### Section 4.6 – Separate special StockCodes

In [6]:
# Identify special StockCodes
special_codes = ["POST", "DOT", "M"]
special_code_records = valid_positive_transactions[
    valid_positive_transactions["StockCode"].isin(special_codes)
].copy()

special_code_summary = (
    special_code_records.groupby("StockCode")
    .agg(
        Records=("StockCode", "size"),
        Customers=("CustomerID", "nunique"),
        Invoices=("InvoiceNo", "nunique"),
        Total_Quantity=("Quantity", "sum")
    )
)

print("SPECIAL STOCKCODE CHECK")
print("Valid positive transactions:", len(valid_positive_transactions))
display(special_code_summary)


SPECIAL STOCKCODE CHECK
Valid positive transactions: 392692


,Records,Customers,Invoices,Total_Quantity
StockCode,,,,
DOT,16,1,16,16
M,279,197,253,6933
POST,1099,331,1099,3120


**Interpretation**

`POST`, `DOT` and `M` are isolated as special transaction codes rather than treated as ordinary merchandise. Other alphanumeric StockCodes are not removed solely because they contain letters.

### 4.7 – Calculate transaction values and create merchandise datasets

In [7]:
# Calculate transaction value

valid_positive_transactions = valid_positive_transactions.copy()

valid_positive_transactions["LineValue"] = (
    valid_positive_transactions["Quantity"]
    * valid_positive_transactions["UnitPrice"]
)

# Dataset 1: Merchandise only (primary segmentation dataset)
merchandise_df = valid_positive_transactions[
    ~valid_positive_transactions["StockCode"].isin(["POST", "DOT", "M"])
].copy()

# Dataset 2: Merchandise + postage (supplementary analysis)
merchandise_plus_postage_df = valid_positive_transactions[
    ~valid_positive_transactions["StockCode"].isin(["DOT", "M"])
].copy()

# Dataset 3: Special codes retained separately
special_code_records = valid_positive_transactions[
    valid_positive_transactions["StockCode"].isin(["POST", "DOT", "M"])
].copy()

print("DATASET CREATION SUMMARY")

print("\nValid positive transactions:", len(valid_positive_transactions))

print("\nMerchandise-only dataset:")
print("Records:", len(merchandise_df))
print("Customers:", merchandise_df["CustomerID"].nunique())
print("Invoices:", merchandise_df["InvoiceNo"].nunique())
print("Products:", merchandise_df["StockCode"].nunique())
print("Merchandise value: £{:,.2f}".format(merchandise_df["LineValue"].sum()))

print("\nMerchandise + postage dataset:")
print("Records:", len(merchandise_plus_postage_df))
print("Customers:", merchandise_plus_postage_df["CustomerID"].nunique())
print("Invoices:", merchandise_plus_postage_df["InvoiceNo"].nunique())
print("Total value: £{:,.2f}".format(merchandise_plus_postage_df["LineValue"].sum()))

print("\nSpecial-code records:")
print("Records:", len(special_code_records))
print("Total value: £{:,.2f}".format(special_code_records["LineValue"].sum()))

print("\nValidation:")
print(
    len(merchandise_df) + len(special_code_records)
    == len(valid_positive_transactions)
)

DATASET CREATION SUMMARY

Valid positive transactions: 392692

Merchandise-only dataset:
Records: 391298
Customers: 4335
Invoices: 18416
Products: 3662
Merchandise value: £8,744,078.64

Merchandise + postage dataset:
Records: 392397
Customers: 4335
Invoices: 18468
Total value: £8,821,882.60

Special-code records:
Records: 1394
Total value: £143,130.25

Validation:
True


**Interpretation**

Transaction line value is calculated as Quantity multiplied by UnitPrice. The primary dataset excludes `POST`, `DOT` and `M`; a supplementary merchandise-plus-postage dataset is retained.

### Section 4.8 – Define the analysis reference date

In [8]:
# Define the reference date for RFM calculations

last_transaction_date = merchandise_df["InvoiceDate"].max()

reference_date = last_transaction_date.normalize() + pd.Timedelta(days=1)

print("Last transaction date:", last_transaction_date)
print("RFM reference date:", reference_date)
print("Reference date is after the final transaction:",
      reference_date > last_transaction_date)

Last transaction date: 2011-12-09 12:50:00
RFM reference date: 2011-12-10 00:00:00
Reference date is after the final transaction: True


**Interpretation**

The reference date is set to 10 December 2011, one calendar day after the final recorded transaction date. This fixed point ensures Recency is calculated consistently.

### Section 4.9 – Create customer-level RFM features

In [9]:
# Create core RFM features at customer level

rfm_df = (
    merchandise_df
    .groupby("CustomerID")
    .agg(
        LastPurchaseDate=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("LineValue", "sum")
    )
    .reset_index()
)

# Calculate Recency in days
rfm_df["Recency"] = (
    reference_date - rfm_df["LastPurchaseDate"].dt.normalize()
).dt.days

# Arrange columns
rfm_df = rfm_df[
    ["CustomerID", "Recency", "Frequency", "Monetary", "LastPurchaseDate"]
]

print("RFM FEATURE SUMMARY")
print("Number of customers:", rfm_df["CustomerID"].nunique())
print("RFM dataset shape:", rfm_df.shape)

display(rfm_df.head())

print("\nRFM descriptive statistics:")
display(
    rfm_df[["Recency", "Frequency", "Monetary"]]
    .describe()
    .round(2)
)

RFM FEATURE SUMMARY
Number of customers: 4335
RFM dataset shape: (4335, 5)


,CustomerID,Recency,Frequency,Monetary,LastPurchaseDate
0,12346.0,326,1,77183.60,2011-01-18 10:01:00
1,12347.0,3,7,4310.00,2011-12-07 15:52:00
2,12348.0,76,4,1437.24,2011-09-25 13:13:00
3,12349.0,19,1,1457.55,2011-11-21 09:51:00
4,12350.0,311,1,294.40,2011-02-02 16:01:00



RFM descriptive statistics:


,Recency,Frequency,Monetary
count,4335.00,4335.00,4335.00
mean,93.23,4.25,2017.09
std,100.20,7.65,8919.39
min,1.00,1.00,3.75
25%,18.00,1.00,304.25
50%,51.00,2.00,663.61
75%,143.00,5.00,1631.48
max,374.00,206.00,279138.02


### Section 4.10 – Extended behavioural features

In [10]:
# Create extended customer-level behavioural features

# 1. Average Order Value
rfm_df["AverageOrderValue"] = (
    rfm_df["Monetary"] / rfm_df["Frequency"]
)

# 2. Total Quantity and 3. Product Diversity
customer_behaviour = (
    merchandise_df
    .groupby("CustomerID")
    .agg(
        TotalQuantity=("Quantity", "sum"),
        ProductDiversity=("StockCode", "nunique")
    )
    .reset_index()
)

# 4. Purchase Regularity
# First, obtain one purchase date per customer per invoice
invoice_dates = (
    merchandise_df
    .groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"]
    .min()
    .reset_index()
    .sort_values(["CustomerID", "InvoiceDate"])
)

# Calculate the gap in days between consecutive invoices
invoice_dates["DaysBetweenPurchases"] = (
    invoice_dates
    .groupby("CustomerID")["InvoiceDate"]
    .diff()
    .dt.total_seconds() / (24 * 60 * 60)
)

purchase_regularity = (
    invoice_dates
    .groupby("CustomerID")["DaysBetweenPurchases"]
    .mean()
    .reset_index(name="PurchaseRegularity")
)

# Merge extended features into the RFM dataset
rfm_df = (
    rfm_df
    .merge(customer_behaviour, on="CustomerID", how="left")
    .merge(purchase_regularity, on="CustomerID", how="left")
)

print("EXTENDED FEATURE SUMMARY")
print("Dataset shape:", rfm_df.shape)
print("Number of customers:", rfm_df["CustomerID"].nunique())

display(rfm_df.head())

print("\nExtended feature descriptive statistics:")
display(
    rfm_df[
        [
            "AverageOrderValue",
            "TotalQuantity",
            "ProductDiversity",
            "PurchaseRegularity"
        ]
    ].describe().round(2)
)

print("\nMissing Purchase Regularity values:",
      rfm_df["PurchaseRegularity"].isna().sum())

EXTENDED FEATURE SUMMARY
Dataset shape: (4335, 9)
Number of customers: 4335


,CustomerID,Recency,Frequency,Monetary,LastPurchaseDate,AverageOrderValue,TotalQuantity,ProductDiversity,PurchaseRegularity
0,12346.0,326,1,77183.60,2011-01-18 10:01:00,77183.600000,74215,1,NaN
1,12347.0,3,7,4310.00,2011-12-07 15:52:00,615.714286,2458,103,60.839699
2,12348.0,76,4,1437.24,2011-09-25 13:13:00,359.310000,2332,21,94.250926
3,12349.0,19,1,1457.55,2011-11-21 09:51:00,1457.550000,630,72,NaN
4,12350.0,311,1,294.40,2011-02-02 16:01:00,294.400000,196,16,NaN



Extended feature descriptive statistics:


,AverageOrderValue,TotalQuantity,ProductDiversity,PurchaseRegularity
count,4335.00,4335.00,4335.00,2830.00
mean,415.47,1186.14,61.42,73.09
std,1800.67,5040.41,85.31,65.53
min,3.75,1.00,1.00,0.00
25%,177.09,159.00,16.00,30.08
50%,289.65,377.00,35.00,54.03
75%,423.37,989.50,77.00,92.70
max,84236.25,196844.00,1786.00,365.98



Missing Purchase Regularity values: 1505


**Interpretation**

Purchase Regularity is the mean number of days between consecutive purchase invoices. It is naturally missing for single-invoice customers, is not imputed, and is excluded from clustering.

### Section 4.11 – Validate Purchase Regularity

In [11]:
# Validate missing Purchase Regularity values

missing_regularity = rfm_df[
    rfm_df["PurchaseRegularity"].isna()
]

multiple_invoice_missing = missing_regularity[
    missing_regularity["Frequency"] > 1
]

single_invoice_customers = rfm_df[
    rfm_df["Frequency"] == 1
]

print("PURCHASE REGULARITY VALIDATION")

print(
    "Customers with missing Purchase Regularity:",
    len(missing_regularity)
)

print(
    "Customers with only one invoice:",
    len(single_invoice_customers)
)

print(
    "Customers with missing regularity but multiple invoices:",
    len(multiple_invoice_missing)
)

print(
    "\nValidation:",
    len(missing_regularity) == len(single_invoice_customers)
)

print("\nFrequency distribution:")
display(rfm_df["Frequency"].value_counts().sort_index().head(10))

PURCHASE REGULARITY VALIDATION
Customers with missing Purchase Regularity: 1505
Customers with only one invoice: 1505
Customers with missing regularity but multiple invoices: 0

Validation: True

Frequency distribution:


Frequency
1     1505
2      830
3      504
4      394
5      237
6      173
7      138
8       98
9       69
10      55
Name: count, dtype: int64

### Section 4.12 – Check customer-level feature quality

In [12]:
# Final customer-level feature quality check

feature_columns = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity",
    "PurchaseRegularity"
]

print("CUSTOMER FEATURE QUALITY CHECK")

print("\nMissing values:")
display(rfm_df[feature_columns].isna().sum())

print("\nInfinite values:")
display(
    pd.Series(
        np.isinf(rfm_df[feature_columns]).sum(),
        name="Infinite Values"
    )
)

print("\nDuplicate CustomerIDs:", rfm_df["CustomerID"].duplicated().sum())

print("\nNegative values:")
display((rfm_df[feature_columns] < 0).sum())

print("\nFinal customer dataset shape:", rfm_df.shape)

CUSTOMER FEATURE QUALITY CHECK

Missing values:


Recency                  0
Frequency                0
Monetary                 0
AverageOrderValue        0
TotalQuantity            0
ProductDiversity         0
PurchaseRegularity    1505
dtype: int64


Infinite values:


Recency               0
Frequency             0
Monetary              0
AverageOrderValue     0
TotalQuantity         0
ProductDiversity      0
PurchaseRegularity    0
Name: Infinite Values, dtype: int64


Duplicate CustomerIDs: 0

Negative values:


Recency               0
Frequency             0
Monetary              0
AverageOrderValue     0
TotalQuantity         0
ProductDiversity      0
PurchaseRegularity    0
dtype: int64


Final customer dataset shape: (4335, 9)


### Section 4.13 – Examine skewness and extreme values

In [13]:
# Examine skewness and extreme values in customer features

clustering_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

feature_diagnostics = []

for col in clustering_features:
    series = rfm_df[col]

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    feature_diagnostics.append({
        "Feature": col,
        "Skewness": series.skew(),
        "Minimum": series.min(),
        "Median": series.median(),
        "95th Percentile": series.quantile(0.95),
        "99th Percentile": series.quantile(0.99),
        "Maximum": series.max(),
        "IQR Outlier Count": (
            (series < lower_bound) | (series > upper_bound)
        ).sum()
    })

diagnostics_df = pd.DataFrame(feature_diagnostics)

display(diagnostics_df.round(2))

,Feature,Skewness,Minimum,Median,95th Percentile,99th Percentile,Maximum,IQR Outlier Count
0,Recency,1.24,1.00,51.00,312.00,369.66,374.00,154
1,Frequency,11.97,1.00,2.00,13.00,30.00,206.00,278
2,Monetary,19.55,3.75,663.61,5734.89,18714.20,279138.02,421
3,AverageOrderValue,41.46,3.75,289.65,915.28,2023.73,84236.25,293
4,TotalQuantity,20.42,1.00,377.00,3554.60,10888.08,196844.00,400
5,ProductDiversity,6.92,1.00,35.00,204.00,353.66,1786.00,309


**Interpretation**

The customer features are strongly right-skewed, particularly Monetary (skewness 19.55), Total Quantity (20.42) and Average Order Value (41.46). IQR flags are diagnostic indicators, not automatic grounds for deleting customers.

### Section 4.14 – Evaluate log transformations

In [14]:
# Compare skewness before and after log1p transformation

features_to_transform = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

skewness_comparison = []

for col in features_to_transform:
    original_skew = rfm_df[col].skew()
    transformed_skew = np.log1p(rfm_df[col]).skew()

    skewness_comparison.append({
        "Feature": col,
        "Original Skewness": original_skew,
        "Log1p Skewness": transformed_skew,
        "Change in Skewness": original_skew - transformed_skew
    })

skewness_comparison_df = pd.DataFrame(skewness_comparison)

display(skewness_comparison_df.round(3))

,Feature,Original Skewness,Log1p Skewness,Change in Skewness
0,Recency,1.244,-0.327,1.570
1,Frequency,11.970,1.214,10.756
2,Monetary,19.554,0.395,19.158
3,AverageOrderValue,41.462,0.271,41.191
4,TotalQuantity,20.415,0.074,20.341
5,ProductDiversity,6.920,-0.244,7.164


**Interpretation**

Log1p substantially reduces skewness for Frequency, Monetary, Average Order Value, Total Quantity and Product Diversity. Recency also becomes less skewed under log1p, but is deliberately retained in its original day scale.

### Section 4.15 – Create the transformed feature dataset

In [15]:
# Create transformed clustering features

transformed_df = rfm_df[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "AverageOrderValue",
        "TotalQuantity",
        "ProductDiversity"
    ]
].copy()

# Apply log1p to selected right-skewed variables
log_features = [
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

for col in log_features:
    transformed_df[col] = np.log1p(transformed_df[col])

print("TRANSFORMED FEATURE DATASET")
print("Shape:", transformed_df.shape)

display(transformed_df.head())

print("\nDescriptive statistics after transformation:")
display(
    transformed_df.drop(columns="CustomerID")
    .describe()
    .round(3)
)

print("\nMissing values:")
display(transformed_df.isna().sum())

TRANSFORMED FEATURE DATASET
Shape: (4335, 7)


,CustomerID,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,ProductDiversity
0,12346.0,326,0.693147,11.253955,11.253955,11.214735,0.693147
1,12347.0,3,2.079442,8.368925,6.424406,7.807510,4.644391
2,12348.0,76,1.609438,7.271175,5.886965,7.754910,3.091042
3,12349.0,19,0.693147,7.285198,7.285198,6.447306,4.290459
4,12350.0,311,0.693147,5.688330,5.688330,5.283204,2.833213



Descriptive statistics after transformation:


,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,ProductDiversity
count,4335.000,4335.000,4335.000,4335.000,4335.000,4335.000
mean,93.233,1.342,6.575,5.638,5.983,3.549
std,100.204,0.682,1.256,0.745,1.365,1.133
min,1.000,0.693,1.558,1.558,0.693,0.693
25%,18.000,0.693,5.721,5.182,5.075,2.833
50%,51.000,1.099,6.499,5.672,5.935,3.584
75%,143.000,1.792,7.398,6.051,6.898,4.357
max,374.000,5.333,12.539,11.341,12.190,7.488



Missing values:


CustomerID           0
Recency              0
Frequency            0
Monetary             0
AverageOrderValue    0
TotalQuantity        0
ProductDiversity     0
dtype: int64

###  Section 4.16 – Feature correlation analysis

In [16]:
# Section 4.16 – Feature correlation analysis

# Select the six transformed clustering features
correlation_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

correlation_df = transformed_df[correlation_features].copy()

# Pearson correlation
pearson_corr = correlation_df.corr(method="pearson")

# Spearman correlation
spearman_corr = correlation_df.corr(method="spearman")

print("PEARSON CORRELATION MATRIX")
display(pearson_corr.round(3))

print("\nSPEARMAN CORRELATION MATRIX")
display(spearman_corr.round(3))

PEARSON CORRELATION MATRIX


,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,ProductDiversity
Recency,1.000,-0.473,-0.435,-0.138,-0.454,-0.419
Frequency,-0.473,1.000,0.808,0.162,0.752,0.621
Monetary,-0.435,0.808,1.000,0.708,0.927,0.694
AverageOrderValue,-0.138,0.162,0.708,1.000,0.650,0.407
TotalQuantity,-0.454,0.752,0.927,0.650,1.000,0.700
ProductDiversity,-0.419,0.621,0.694,0.407,0.700,1.000



SPEARMAN CORRELATION MATRIX


,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,ProductDiversity
Recency,1.000,-0.564,-0.482,-0.126,-0.486,-0.457
Frequency,-0.564,1.000,0.806,0.155,0.759,0.655
Monetary,-0.482,0.806,1.000,0.676,0.930,0.725
AverageOrderValue,-0.126,0.155,0.676,1.000,0.620,0.412
TotalQuantity,-0.486,0.759,0.930,0.620,1.000,0.723
ProductDiversity,-0.457,0.655,0.725,0.412,0.723,1.000


In [17]:
# Identify highly correlated feature pairs

upper_triangle = np.triu(
    np.ones(pearson_corr.shape),
    k=1
).astype(bool)

correlation_pairs = []

for i in range(len(correlation_features)):
    for j in range(i + 1, len(correlation_features)):

        feature_1 = correlation_features[i]
        feature_2 = correlation_features[j]

        correlation_pairs.append({
            "Feature 1": feature_1,
            "Feature 2": feature_2,
            "Pearson Correlation": pearson_corr.loc[feature_1, feature_2],
            "Spearman Correlation": spearman_corr.loc[feature_1, feature_2]
        })

correlation_pairs_df = pd.DataFrame(correlation_pairs)

correlation_pairs_df["Absolute Pearson"] = (
    correlation_pairs_df["Pearson Correlation"].abs()
)

correlation_pairs_df = correlation_pairs_df.sort_values(
    "Absolute Pearson",
    ascending=False
)

display(
    correlation_pairs_df.drop(columns="Absolute Pearson").round(3)
)

,Feature 1,Feature 2,Pearson Correlation,Spearman Correlation
10,Monetary,TotalQuantity,0.927,0.930
5,Frequency,Monetary,0.808,0.806
7,Frequency,TotalQuantity,0.752,0.759
9,Monetary,AverageOrderValue,0.708,0.676
14,TotalQuantity,ProductDiversity,0.700,0.723
11,Monetary,ProductDiversity,0.694,0.725
12,AverageOrderValue,TotalQuantity,0.650,0.620
8,Frequency,ProductDiversity,0.621,0.655
0,Recency,Frequency,-0.473,-0.564
3,Recency,TotalQuantity,-0.454,-0.486



### Section 4.17 – Outlier assessment after log transformation

In [18]:
# Section 4.17 – Outlier assessment after log transformation

outlier_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

outlier_comparison = []

for col in outlier_features:

    # Original feature values
    original = rfm_df[col]

    # Transformed feature values
    transformed = transformed_df[col]

    # Original IQR boundaries
    q1_original = original.quantile(0.25)
    q3_original = original.quantile(0.75)
    iqr_original = q3_original - q1_original

    lower_original = q1_original - 1.5 * iqr_original
    upper_original = q3_original + 1.5 * iqr_original

    original_outliers = (
        (original < lower_original) |
        (original > upper_original)
    ).sum()

    # Transformed IQR boundaries
    q1_transformed = transformed.quantile(0.25)
    q3_transformed = transformed.quantile(0.75)
    iqr_transformed = q3_transformed - q1_transformed

    lower_transformed = q1_transformed - 1.5 * iqr_transformed
    upper_transformed = q3_transformed + 1.5 * iqr_transformed

    transformed_outliers = (
        (transformed < lower_transformed) |
        (transformed > upper_transformed)
    ).sum()

    outlier_comparison.append({
        "Feature": col,
        "Original IQR Outliers": original_outliers,
        "Transformed IQR Outliers": transformed_outliers,
        "Reduction in Outliers": (
            original_outliers - transformed_outliers
        )
    })

outlier_comparison_df = pd.DataFrame(outlier_comparison)

display(outlier_comparison_df)

,Feature,Original IQR Outliers,Transformed IQR Outliers,Reduction in Outliers
0,Recency,154,154,0
1,Frequency,278,42,236
2,Monetary,421,53,368
3,AverageOrderValue,293,132,161
4,TotalQuantity,400,64,336
5,ProductDiversity,309,6,303


**Interpretation**

Log transformation substantially reduces IQR-flagged observations for most transformed features. Recency's flagged count remains unchanged because it is not transformed in the modelling dataset. Statistical outliers are retained.

### Section 4.18 – Identify extreme customers after transformation

In [19]:
# Section 4.18 – Identify extreme customers after transformation

extreme_customer_summary = transformed_df[
    ["CustomerID"] + outlier_features
].copy()

print("TOP 10 CUSTOMERS BY TRANSFORMED MONETARY VALUE")

display(
    extreme_customer_summary
    .nlargest(10, "Monetary")
    .merge(
        rfm_df[
            [
                "CustomerID",
                "Frequency",
                "Monetary",
                "TotalQuantity",
                "ProductDiversity"
            ]
        ],
        on="CustomerID",
        suffixes=("_transformed", "_original")
    )
    .round(3)
)

print("\nTOP 10 CUSTOMERS BY TRANSFORMED AVERAGE ORDER VALUE")

display(
    extreme_customer_summary
    .nlargest(10, "AverageOrderValue")
    .merge(
        rfm_df[
            [
                "CustomerID",
                "Frequency",
                "Monetary",
                "AverageOrderValue"
            ]
        ],
        on="CustomerID",
        suffixes=("_transformed", "_original")
    )
    .round(3)
)

TOP 10 CUSTOMERS BY TRANSFORMED MONETARY VALUE


,CustomerID,Recency,Frequency_transformed,Monetary_transformed,AverageOrderValue,TotalQuantity_transformed,ProductDiversity_transformed,Frequency_original,Monetary_original,TotalQuantity_original,ProductDiversity_original
0,14646.0,2,4.290,12.539,8.263,12.190,6.551,72,279138.02,196844,699
1,18102.0,1,4.111,12.467,8.373,11.069,5.017,60,259657.30,64124,150
2,17450.0,9,3.850,12.178,8.349,11.156,4.828,46,194390.79,69973,124
3,16446.0,1,1.099,12.035,11.341,11.302,1.386,2,168472.50,80997,3
4,14911.0,2,5.298,11.852,6.560,11.293,7.488,199,140336.83,80238,1786
5,12415.0,25,3.045,11.733,8.737,11.256,6.096,20,124564.53,77373,443
6,14156.0,10,4.025,11.672,7.665,10.964,6.572,55,117210.08,57768,714
7,17511.0,3,3.466,11.419,7.986,11.075,6.118,31,91062.38,64549,453
8,12346.0,326,0.693,11.254,11.254,11.215,0.693,1,77183.60,74215,1
9,16029.0,39,4.143,11.194,7.068,10.599,3.784,62,72708.09,40107,43



TOP 10 CUSTOMERS BY TRANSFORMED AVERAGE ORDER VALUE


,CustomerID,Recency,Frequency_transformed,Monetary_transformed,AverageOrderValue_transformed,TotalQuantity,ProductDiversity,Frequency_original,Monetary_original,AverageOrderValue_original
0,16446.0,1,1.099,12.035,11.341,11.302,1.386,2,168472.50,84236.250
1,12346.0,326,0.693,11.254,11.254,11.215,0.693,1,77183.60,77183.600
2,15749.0,236,1.386,10.704,9.605,9.800,1.792,3,44534.30,14844.767
3,15098.0,183,1.386,10.595,9.496,4.804,0.693,3,39916.50,13305.500
4,12590.0,212,0.693,9.142,9.142,8.362,4.220,1,9341.26,9341.260
5,12415.0,25,3.045,11.733,8.737,11.256,6.096,20,124564.53,6228.226
6,12357.0,34,0.693,8.734,8.734,7.904,4.883,1,6207.67,6207.670
7,12688.0,114,0.693,8.492,8.492,8.016,5.147,1,4873.81,4873.810
8,12752.0,82,0.693,8.382,8.382,7.724,3.989,1,4366.78,4366.780
9,18102.0,1,4.111,12.467,8.373,11.069,5.017,60,259657.30,4327.622


In [20]:
# Inspect transaction-level records for flagged customers

flagged_customers = [16446, 12346]

# Purchase records
flagged_purchases = deduplicated_df[
    deduplicated_df["CustomerID"].isin(flagged_customers)
    & ~deduplicated_df["InvoiceNo"].astype("string").str.startswith("C", na=False)
].copy()

flagged_purchases["LineValue"] = (
    flagged_purchases["Quantity"]
    * flagged_purchases["UnitPrice"]
)

print("PURCHASE TRANSACTIONS")

display(
    flagged_purchases[
        [
            "CustomerID",
            "InvoiceNo",
            "InvoiceDate",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "LineValue"
        ]
    ].sort_values(["CustomerID", "InvoiceDate"])
)

# Cancellation records for the same customers
flagged_cancellations = cancellation_records[
    cancellation_records["CustomerID"].isin(flagged_customers)
].copy()

print("\nCANCELLATION TRANSACTIONS")

display(
    flagged_cancellations[
        [
            "CustomerID",
            "InvoiceNo",
            "InvoiceDate",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice"
        ]
    ].sort_values(["CustomerID", "InvoiceDate"])
)

PURCHASE TRANSACTIONS


,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,LineValue
61619,12346.0,541431,2011-01-18 10:01:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60
194354,16446.0,553573,2011-05-18 09:52:00,22980,PANTRY SCRUBBING BRUSH,1,1.65,1.65
194355,16446.0,553573,2011-05-18 09:52:00,22982,PANTRY PASTRY BRUSH,1,1.25,1.25
540421,16446.0,581483,2011-12-09 09:15:00,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60



CANCELLATION TRANSACTIONS


,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice
61624,12346.0,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04
540422,16446.0,C581484,2011-12-09 09:27:00,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08


**Interpretation**

The transaction-level inspection identifies two strongly matched purchase–cancellation pairs: Customer 12346's 74,215-unit purchase and reversal, and Customer 16446's 80,995-unit purchase and reversal. Each cancellation follows its purchase by approximately 12–16 minutes and matches on customer, product, quantity and unit price.

In [21]:
# Section 4.18.2 – Create matched-reversal sensitivity dataset

# Identify the two confirmed matched purchase lines
matched_purchase_mask = (
    (
        (merchandise_df["CustomerID"] == 12346)
        & (merchandise_df["InvoiceNo"] == "541431")
        & (merchandise_df["StockCode"] == "23166")
        & (merchandise_df["Quantity"] == 74215)
        & (merchandise_df["UnitPrice"] == 1.04)
    )
    |
    (
        (merchandise_df["CustomerID"] == 16446)
        & (merchandise_df["InvoiceNo"] == "581483")
        & (merchandise_df["StockCode"] == "23843")
        & (merchandise_df["Quantity"] == 80995)
        & (merchandise_df["UnitPrice"] == 2.08)
    )
)

matched_reversal_lines = merchandise_df[
    matched_purchase_mask
].copy()

# Create sensitivity dataset excluding only the matched lines
merchandise_sensitivity_df = merchandise_df[
    ~matched_purchase_mask
].copy()

print("MATCHED REVERSAL SENSITIVITY CHECK")

print("\nMatched purchase lines identified:")
display(
    matched_reversal_lines[
        [
            "CustomerID",
            "InvoiceNo",
            "StockCode",
            "Quantity",
            "UnitPrice",
            "LineValue"
        ]
    ]
)

print("\nOriginal merchandise dataset:")
print("Records:", len(merchandise_df))
print("Customers:", merchandise_df["CustomerID"].nunique())
print("Total value: £{:,.2f}".format(merchandise_df["LineValue"].sum()))

print("\nSensitivity dataset:")
print("Records:", len(merchandise_sensitivity_df))
print("Customers:", merchandise_sensitivity_df["CustomerID"].nunique())
print(
    "Total value: £{:,.2f}".format(
        merchandise_sensitivity_df["LineValue"].sum()
    )
)

print("\nMatched purchase value: £{:,.2f}".format(
    matched_reversal_lines["LineValue"].sum()
))

print("\nValidation:")
print(
    len(merchandise_df)
    - len(matched_reversal_lines)
    == len(merchandise_sensitivity_df)
)

MATCHED REVERSAL SENSITIVITY CHECK

Matched purchase lines identified:


,CustomerID,InvoiceNo,StockCode,Quantity,UnitPrice,LineValue
61619,12346.0,541431,23166,74215,1.04,77183.6
540421,16446.0,581483,23843,80995,2.08,168469.6



Original merchandise dataset:
Records: 391298
Customers: 4335
Total value: £8,744,078.64

Sensitivity dataset:
Records: 391296
Customers: 4334
Total value: £8,498,425.44

Matched purchase value: £245,653.20

Validation:
True


**Interpretation**

Only the two identified purchase lines are removed in the sensitivity dataset; entire invoices and customers are not indiscriminately deleted. The original merchandise dataset contains 4,335 customers and £8,744,078.64 in merchandise value. The sensitivity dataset contains 4,334 customers and £8,498,425.44.

In [22]:
# Recalculate customer-level features for sensitivity analysis

sensitivity_rfm = (
    merchandise_sensitivity_df
    .groupby("CustomerID")
    .agg(
        LastPurchaseDate=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("LineValue", "sum"),
        TotalQuantity=("Quantity", "sum"),
        ProductDiversity=("StockCode", "nunique")
    )
    .reset_index()
)

# Calculate Recency
sensitivity_rfm["Recency"] = (
    reference_date
    - sensitivity_rfm["LastPurchaseDate"].dt.normalize()
).dt.days

# Calculate Average Order Value
sensitivity_rfm["AverageOrderValue"] = (
    sensitivity_rfm["Monetary"]
    / sensitivity_rfm["Frequency"]
)

# Apply the same log transformations
sensitivity_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

sensitivity_transformed = sensitivity_rfm[
    ["CustomerID"] + sensitivity_features
].copy()

log_features = [
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

for col in log_features:
    sensitivity_transformed[col] = np.log1p(
        sensitivity_transformed[col]
    )

print("SENSITIVITY CUSTOMER DATASET")
print("Customers:", len(sensitivity_rfm))
print("Dataset shape:", sensitivity_rfm.shape)

print("\nCustomer 16446:")
display(
    sensitivity_rfm[
        sensitivity_rfm["CustomerID"] == 16446
    ]
)

print("\nCustomer 12346:")
display(
    sensitivity_rfm[
        sensitivity_rfm["CustomerID"] == 12346
    ]
)

print("\nMissing values in transformed features:")
display(sensitivity_transformed.isna().sum())

SENSITIVITY CUSTOMER DATASET
Customers: 4334
Dataset shape: (4334, 8)

Customer 16446:


,CustomerID,LastPurchaseDate,Frequency,Monetary,TotalQuantity,ProductDiversity,Recency,AverageOrderValue
3006,16446.0,2011-05-18 09:52:00,1,2.9,2,2,206,2.9



Customer 12346:


,CustomerID,LastPurchaseDate,Frequency,Monetary,TotalQuantity,ProductDiversity,Recency,AverageOrderValue



Missing values in transformed features:


CustomerID           0
Recency              0
Frequency            0
Monetary             0
AverageOrderValue    0
TotalQuantity        0
ProductDiversity     0
dtype: int64

In [23]:
# Section 4.18.4 – Compare original and sensitivity customer features

comparison_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

original_summary = (
    rfm_df[comparison_features]
    .describe()
    .loc[["mean", "50%", "std", "min", "max"]]
    .T
)

sensitivity_summary = (
    sensitivity_rfm[comparison_features]
    .describe()
    .loc[["mean", "50%", "std", "min", "max"]]
    .T
)

feature_comparison = original_summary.add_suffix("_Original").join(
    sensitivity_summary.add_suffix("_Sensitivity")
)

display(feature_comparison.round(2))

,mean_Original,50%_Original,std_Original,min_Original,max_Original,mean_Sensitivity,50%_Sensitivity,std_Sensitivity,min_Sensitivity,max_Sensitivity
Recency,93.23,51.00,100.20,1.00,374.00,93.23,51.00,100.16,1.0,374.00
Frequency,4.25,2.00,7.65,1.00,206.00,4.25,2.00,7.65,1.0,206.00
Monetary,2017.09,663.61,8919.39,3.75,279138.02,1960.87,661.42,8477.81,2.9,279138.02
AverageOrderValue,415.47,289.65,1800.67,3.75,84236.25,378.33,289.46,510.11,2.9,14844.77
TotalQuantity,1186.14,377.00,5040.41,1.00,196844.00,1150.61,376.50,4765.47,1.0,196844.00
ProductDiversity,61.42,35.00,85.31,1.00,1786.00,61.44,35.00,85.32,1.0,1786.00


**Interpretation**

Removing the two matched purchase lines has a limited effect on overall customer-level feature distributions, although maximum and mean Monetary, Average Order Value and Total Quantity decline.

### Section 4.19 – Define feature sets for clustering

In [24]:
# Section 4.19 – Define feature sets for clustering

rfm_features = [
    "Recency",
    "Frequency",
    "Monetary"
]

extended_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "ProductDiversity"
]

reduced_extended_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "ProductDiversity"
]

# Validate feature availability
for feature_set_name, features in {
    "RFM": rfm_features,
    "Extended": extended_features,
    "Reduced Extended": reduced_extended_features
}.items():
    missing_features = [
        col for col in features
        if col not in transformed_df.columns
    ]

    print(f"\n{feature_set_name}")
    print("Features:", features)
    print("Number of features:", len(features))
    print("Missing columns:", missing_features)

# Create separate modelling datasets
rfm_model_df = transformed_df[
    ["CustomerID"] + rfm_features
].copy()

extended_model_df = transformed_df[
    ["CustomerID"] + extended_features
].copy()

reduced_extended_model_df = transformed_df[
    ["CustomerID"] + reduced_extended_features
].copy()

print("\nMODELLING DATASET SHAPES")
print("RFM:", rfm_model_df.shape)
print("Extended:", extended_model_df.shape)
print("Reduced Extended:", reduced_extended_model_df.shape)


RFM
Features: ['Recency', 'Frequency', 'Monetary']
Number of features: 3
Missing columns: []

Extended
Features: ['Recency', 'Frequency', 'Monetary', 'AverageOrderValue', 'TotalQuantity', 'ProductDiversity']
Number of features: 6
Missing columns: []

Reduced Extended
Features: ['Recency', 'Frequency', 'Monetary', 'AverageOrderValue', 'ProductDiversity']
Number of features: 5
Missing columns: []

MODELLING DATASET SHAPES
RFM: (4335, 4)
Extended: (4335, 7)
Reduced Extended: (4335, 6)


**Interpretation**

Three feature sets are prepared: RFM (3 features), Extended (6 features) and Reduced Extended (5 features). CustomerID is retained for mapping results back to customers, but excluded from clustering inputs.

### Section 4.20 – Apply both scaling methods

In [25]:
# Section 4.20 – Compare StandardScaler and RobustScaler

from sklearn.preprocessing import StandardScaler, RobustScaler

# Feature sets (exclude CustomerID)
feature_sets = {
    "RFM": rfm_features,
    "Extended": extended_features,
    "Reduced_Extended": reduced_extended_features
}

# Store scaled datasets
scaled_datasets = {}

# Apply both scalers to every feature set
for set_name, features in feature_sets.items():

    X = transformed_df[features].copy()

    # StandardScaler
    standard_scaler = StandardScaler()
    X_standard = pd.DataFrame(
        standard_scaler.fit_transform(X),
        columns=features,
        index=X.index
    )

    # RobustScaler
    robust_scaler = RobustScaler()
    X_robust = pd.DataFrame(
        robust_scaler.fit_transform(X),
        columns=features,
        index=X.index
    )

    scaled_datasets[f"{set_name}_Standard"] = X_standard
    scaled_datasets[f"{set_name}_Robust"] = X_robust

    print(f"\n{set_name}")
    print("Original feature shape:", X.shape)
    print("StandardScaler shape:", X_standard.shape)
    print("RobustScaler shape:", X_robust.shape)

    print("\nStandardScaler means:")
    display(X_standard.mean().round(3))

    print("StandardScaler standard deviations:")
    display(X_standard.std(ddof=0).round(3))

    print("RobustScaler medians:")
    display(X_robust.median().round(3))

    print("RobustScaler IQR:")
    display(
        (
            X_robust.quantile(0.75)
            - X_robust.quantile(0.25)
        ).round(3)
    )


RFM
Original feature shape: (4335, 3)
StandardScaler shape: (4335, 3)
RobustScaler shape: (4335, 3)

StandardScaler means:


Recency      0.0
Frequency    0.0
Monetary    -0.0
dtype: float64

StandardScaler standard deviations:


Recency      1.0
Frequency    1.0
Monetary     1.0
dtype: float64

RobustScaler medians:


Recency      0.0
Frequency    0.0
Monetary     0.0
dtype: float64

... (14 outputs left out)

RobustScaler medians:


Recency              0.0
Frequency            0.0
Monetary             0.0
AverageOrderValue    0.0
ProductDiversity     0.0
dtype: float64

RobustScaler IQR:


Recency              1.0
Frequency            1.0
Monetary             1.0
AverageOrderValue    1.0
ProductDiversity     1.0
dtype: float64

### Section 4.21 – Prepare sensitivity feature sets

In [26]:
# Section 4.21 – Define sensitivity feature sets

sensitivity_rfm_model_df = sensitivity_transformed[
    ["CustomerID"] + rfm_features
].copy()

sensitivity_extended_model_df = sensitivity_transformed[
    ["CustomerID"] + extended_features
].copy()

sensitivity_reduced_extended_model_df = sensitivity_transformed[
    ["CustomerID"] + reduced_extended_features
].copy()

sensitivity_feature_sets = {
    "RFM": sensitivity_rfm_model_df,
    "Extended": sensitivity_extended_model_df,
    "Reduced_Extended": sensitivity_reduced_extended_model_df
}

print("SENSITIVITY MODELLING DATASETS")

for name, dataset in sensitivity_feature_sets.items():
    print(f"{name}: {dataset.shape}")
    print("Missing values:", dataset.isna().sum().sum())

SENSITIVITY MODELLING DATASETS
RFM: (4334, 4)
Missing values: 0
Extended: (4334, 7)
Missing values: 0
Reduced_Extended: (4334, 6)
Missing values: 0


### Section 4.22 – Scale the sensitivity datasets

In [27]:
# Section 4.22 – Scale sensitivity datasets

sensitivity_scaled_datasets = {}

for set_name, dataset in sensitivity_feature_sets.items():

    # Exclude CustomerID from clustering features
    X = dataset.drop(columns=["CustomerID"]).copy()

    # StandardScaler
    standard_scaler = StandardScaler()
    X_standard = pd.DataFrame(
        standard_scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    # RobustScaler
    robust_scaler = RobustScaler()
    X_robust = pd.DataFrame(
        robust_scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    sensitivity_scaled_datasets[f"{set_name}_Standard"] = X_standard
    sensitivity_scaled_datasets[f"{set_name}_Robust"] = X_robust

    print(f"\n{set_name}")
    print("StandardScaler shape:", X_standard.shape)
    print("RobustScaler shape:", X_robust.shape)

    print("StandardScaler means:")
    display(X_standard.mean().round(3))

    print("RobustScaler medians:")
    display(X_robust.median().round(3))

print("\nTotal sensitivity scaled datasets:",
      len(sensitivity_scaled_datasets))


RFM
StandardScaler shape: (4334, 3)
RobustScaler shape: (4334, 3)
StandardScaler means:


Recency     -0.0
Frequency    0.0
Monetary    -0.0
dtype: float64

RobustScaler medians:


Recency      0.0
Frequency    0.0
Monetary     0.0
dtype: float64


Extended
StandardScaler shape: (4334, 6)
RobustScaler shape: (4334, 6)
StandardScaler means:


Recency             -0.0
Frequency            0.0
Monetary            -0.0
AverageOrderValue    0.0
TotalQuantity        0.0
ProductDiversity     0.0
dtype: float64

... (3 outputs left out)

Recency             -0.0
Frequency            0.0
Monetary            -0.0
AverageOrderValue    0.0
ProductDiversity     0.0
dtype: float64

RobustScaler medians:


Recency              0.0
Frequency            0.0
Monetary             0.0
AverageOrderValue    0.0
ProductDiversity     0.0
dtype: float64


Total sensitivity scaled datasets: 6


**Interpretation**

The same three feature sets are constructed for the sensitivity population of 4,334 customers, with both scaling methods.


### Section 4.23 – Final validation of all modelling datasets

In [28]:
# Section 4.23 – Final validation of all scaled datasets

all_scaled_datasets = {
    **{
        f"Original_{name}": dataset
        for name, dataset in scaled_datasets.items()
    },
    **{
        f"Sensitivity_{name}": dataset
        for name, dataset in sensitivity_scaled_datasets.items()
    }
}

validation_results = []

for name, dataset in all_scaled_datasets.items():

    validation_results.append({
        "Dataset": name,
        "Rows": dataset.shape[0],
        "Features": dataset.shape[1],
        "Missing_Values": dataset.isna().sum().sum(),
        "Infinite_Values": np.isinf(dataset).sum().sum(),
        "CustomerID_Excluded": "CustomerID" not in dataset.columns,
        "Valid": (
            dataset.notna().all().all()
            and np.isfinite(dataset.to_numpy()).all()
            and "CustomerID" not in dataset.columns
        )
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("\nFINAL VALIDATION SUMMARY")
print("Total datasets:", len(validation_df))
print("Valid datasets:", validation_df["Valid"].sum())
print("All datasets valid:", validation_df["Valid"].all())

,Dataset,Rows,Features,Missing_Values,Infinite_Values,CustomerID_Excluded,Valid
0,Original_RFM_Standard,4335,3,0,0,True,True
1,Original_RFM_Robust,4335,3,0,0,True,True
2,Original_Extended_Standard,4335,6,0,0,True,True
3,Original_Extended_Robust,4335,6,0,0,True,True
4,Original_Reduced_Extended_Standard,4335,5,0,0,True,True
5,Original_Reduced_Extended_Robust,4335,5,0,0,True,True
6,Sensitivity_RFM_Standard,4334,3,0,0,True,True
7,Sensitivity_RFM_Robust,4334,3,0,0,True,True
8,Sensitivity_Extended_Standard,4334,6,0,0,True,True
9,Sensitivity_Extended_Robust,4334,6,0,0,True,True



FINAL VALIDATION SUMMARY
Total datasets: 12
Valid datasets: 12
All datasets valid: True


**Interpretation**

All 12 modelling datasets pass validation: six original and six sensitivity datasets, covering three feature sets and two scaling methods. There are no missing or infinite values, and CustomerID is excluded from model inputs.

### Section 4.24 – Save processed datasets

In [29]:
# Section 4.24 – Save all processed datasets

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 1. Customer-level datasets
customer_datasets = {
    "original_customer_features.csv": rfm_df,
    "sensitivity_customer_features.csv": sensitivity_rfm,
    "original_transformed_features.csv": transformed_df,
    "sensitivity_transformed_features.csv": sensitivity_transformed
}

# 2. Original scaled modelling datasets
for name, dataset in scaled_datasets.items():
    customer_datasets[f"original_{name.lower()}_scaled.csv"] = pd.concat(
        [
            transformed_df[["CustomerID"]].reset_index(drop=True),
            dataset.reset_index(drop=True)
        ],
        axis=1
    )

# 3. Sensitivity scaled modelling datasets
for name, dataset in sensitivity_scaled_datasets.items():
    customer_datasets[f"sensitivity_{name.lower()}_scaled.csv"] = pd.concat(
        [
            sensitivity_transformed[["CustomerID"]].reset_index(drop=True),
            dataset.reset_index(drop=True)
        ],
        axis=1
    )

# Save all datasets
saved_files = []

for filename, dataset in customer_datasets.items():
    output_path = processed_dir / filename
    dataset.to_csv(output_path, index=False)

    saved_files.append({
        "File": filename,
        "Rows": len(dataset),
        "Columns": len(dataset.columns),
        "Size_KB": round(output_path.stat().st_size / 1024, 2)
    })

saved_files_df = pd.DataFrame(saved_files)

print("PROCESSED DATASETS SAVED")
display(saved_files_df)

print("\nTotal files saved:", len(saved_files_df))
print("Output directory:", processed_dir)

PROCESSED DATASETS SAVED


,File,Rows,Columns,Size_KB
0,original_customer_features.csv,4335,9,308.77
1,sensitivity_customer_features.csv,4334,8,258.10
2,original_transformed_features.csv,4335,7,438.09
3,sensitivity_transformed_features.csv,4334,7,437.99
4,original_rfm_standard_scaled.csv,4335,4,288.08
5,original_rfm_robust_scaled.csv,4335,4,217.78
6,original_extended_standard_scaled.csv,4335,7,537.72
7,original_extended_robust_scaled.csv,4335,7,467.92
8,original_reduced_extended_standard_scaled.csv,4335,6,454.56
9,original_reduced_extended_robust_scaled.csv,4335,6,384.25



Total files saved: 16
Output directory: c:\Users\yadav\OneDrive\Desktop\Online_Retail_Customer_Segmentation\data\processed


**Interpretation**

Customer-level, transformed and scaled datasets are exported separately to `data/processed/`. CustomerID is included in exported scaled files solely to preserve customer-to-cluster mapping; it is not part of the feature matrix used by the algorithms.

In [30]:
# Section 4.25 – Save preprocessing decision log

decision_log = """
# Phase 3: Data Preprocessing and Feature Engineering

## 1. Dataset preparation
- Original raw dataset preserved unchanged.
- 5,268 exact duplicate records removed from the analytical copy.
- Cancellation invoices (InvoiceNo beginning with C) separated from purchases.
- Records with missing CustomerID excluded from customer-level analysis.
- Only positive Quantity and positive UnitPrice transactions retained.

## 2. Special transaction treatment
- Zero-price transactions excluded from primary paid-purchase features.
- Negative-price accounting adjustments excluded.
- StockCodes POST, DOT and M excluded from primary merchandise features.
- POST retained in a supplementary merchandise-plus-postage dataset.
- Genuine alphanumeric product StockCodes retained.
- Monetary calculated using Quantity multiplied by UnitPrice.

## 3. Customer-level features
Reference date: 10 December 2011.

RFM:
- Recency: days since the customer's last purchase.
- Frequency: number of distinct purchase invoices.
- Monetary: total merchandise expenditure.

Extended features:
- Average Order Value.
- Total Quantity Purchased.
- Product Diversity.
- Purchase Regularity calculated and retained as a supplementary feature.

Purchase Regularity is missing for customers with only one invoice.
It was not imputed and was excluded from primary clustering features.

## 4. Transformations and outliers
- Log1p applied to Frequency, Monetary, Average Order Value,
  Total Quantity and Product Diversity.
- Recency retained in its original scale before standardisation.
- No blanket removal or capping of statistical outliers.
- Extreme customer transactions investigated individually.

## 5. Matched-reversal sensitivity analysis
Two purchase lines were identified with strongly matching cancellation
records based on customer, product, quantity, unit price and timing.

- Customer 12346: purchase value £77,183.60.
- Customer 16446: purchase value £168,469.60.
- Total matched purchase value: £245,653.20.

Both lines were excluded only from the sensitivity dataset.
The original merchandise dataset was retained as the baseline.

## 6. Feature sets
- RFM: Recency, Frequency, Monetary.
- Extended: RFM plus Average Order Value, Total Quantity,
  Product Diversity.
- Reduced Extended: Extended features excluding Total Quantity.

## 7. Scaling
Both StandardScaler and RobustScaler were applied separately
to all three feature sets.

## 8. Final modelling datasets
- Original customer population: 4,335 customers.
- Sensitivity customer population: 4,334 customers.
- Six scaled original datasets.
- Six scaled sensitivity datasets.
- CustomerID retained in exported files for cluster mapping,
  but excluded from clustering model inputs.

## 9. Limitations and considerations
- Cancellation matching is based on strong transaction-level
  evidence but does not establish the full business context.
- Exact duplicate removal may remove legitimate repeated entries
  because the dataset does not provide a unique transaction-line ID.
- Country is reserved for post-clustering profiling.
- Clustering results will be evaluated using internal metrics,
  cluster profiles and interpretability rather than classification accuracy.
"""

report_dir = PROJECT_ROOT / "reports" / "phase_3"
report_dir.mkdir(parents=True, exist_ok=True)

log_path = report_dir / "phase_3_preprocessing_decision_log.md"
log_path.write_text(decision_log.strip(), encoding="utf-8")

print("Preprocessing decision log saved successfully.")
print("Location:", log_path)

Preprocessing decision log saved successfully.
Location: c:\Users\yadav\OneDrive\Desktop\Online_Retail_Customer_Segmentation\reports\phase_3\phase_3_preprocessing_decision_log.md


**Interpretation**

A preprocessing decision log is saved under `reports/phase_3/`, recording cleaning rules, feature definitions, transformations, sensitivity treatment and modelling dataset structure.

## Section 4.26 – Notebook summary and hand-off

### Work completed
- Removed 5,268 excess exact duplicate records from the working copy.
- Separated cancellations, records without CustomerID, invalid values and special StockCodes.
- Created merchandise-only and supplementary merchandise-plus-postage datasets.
- Aggregated 4,335 customers into RFM and extended behavioural features.
- Evaluated Purchase Regularity and excluded it from clustering.
- Applied log1p transformations to selected right-skewed features.
- Examined correlations and defined RFM, Extended and Reduced Extended feature sets.
- Investigated two strongly matched purchase–cancellation pairs and created a separate sensitivity dataset.
- Applied StandardScaler and RobustScaler to original and sensitivity feature sets.
- Validated and exported 12 scaled modelling datasets.

### Key decisions
1. The original raw dataset remains unchanged.
2. The primary clustering dataset is merchandise-only.
3. CustomerID is an identifier, not a clustering feature.
4. Recency remains in days; the other five candidate clustering variables are log1p-transformed.
5. Purchase Regularity is excluded because it is undefined for one-invoice customers.
6. Total Quantity is excluded only from the Reduced Extended experiment.
7. Statistical outliers are not automatically removed or capped.
8. The original customer population is primary; matched-reversal data are used for sensitivity analysis.

### Outputs and transition
Processed datasets are saved under `data/processed/`; the decision log is saved under `reports/phase_3/`. These validated datasets are used in Notebook 05 for K-Means and hierarchical comparison modelling.

*This notebook documents preprocessing and feature construction. Final model evaluation and customer-segment interpretation are addressed in subsequent stages.*
